# Ba baseline còn thiếu - LLMLingua, LongLLMLingua, Selective-Context

Review nói bộ baseline là khoảng trống chính nếu nhắm hội nghị. Ba cái này
nay đã hiện thực trong `run_eval.py`; kernel chạy chúng trên cùng bộ câu hỏi
với các phương pháp đã có để bảng so sánh công bằng.

| baseline | dựa vào gì | có nhìn câu hỏi? |
|---|---|---|
| LLMLingua | perplexity token, LM nhân quả | không |
| LongLLMLingua | xếp hạng đoạn theo câu hỏi + nén không đều | **có** |
| Selective-Context | self-information mức câu | không |

Hai cái "không nhìn câu hỏi" là **đối chứng quan trọng**: chúng tách phần
hiệu quả đến từ *nén nói chung* khỏi phần đến từ *biết câu hỏi là gì*.

Ước tính: LLMLingua 25 s/câu (đo trên CPU, GPU nhanh hơn nhiều), nên n=200
cho ba baseline khoảng 1.5–2h.

**RECOMP nay đã có** - cả hai checkpoint đều công khai trên HuggingFace, chỉ
là không tìm được bằng search, phải biết đúng repo ID.

| RECOMP | kiến trúc | checkpoint |
|---|---|---|
| extractive | sentence-transformers | huấn luyện trên **NQ** (một-hop) - lệch phân bố, phải ghi rõ |
| abstractive | T5 sinh tóm tắt | huấn luyện trên **HotpotQA** - đúng miền multi-hop |

**R2C vẫn chưa** - chưa tìm thấy checkpoint công khai.

In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn từ gist. Repo là private nên Colab không clone ẩn danh được;
# gist thì public, tải thẳng, không phải upload tay lần nào. URL không kèm SHA
# nên luôn trỏ bản mới nhất.
import os, sys, base64, io, zipfile, urllib.request, shutil

# Thư mục làm việc: Colab dùng /content, Kaggle dùng /kaggle/working. Phát hiện
# thay vì cứng hoá - cùng notebook chạy được cả hai, và trên máy khác thì lùi về
# thư mục hiện tại.
BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'repo')
SRC_URL = ('https://gist.githubusercontent.com/nhantrnh/'
           'aeec3610fb9541b5f8b388cafd424eb9/raw/itercomp_src_b64.txt')
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data',
        'metrics', 'fertility', 'stats', 'budget']

def missing_in(root):
    return [m for m in NEED
            if not os.path.exists(os.path.join(root, 'src', 'itercomp', m + '.py'))]

# Ra khỏi REPO trước khi xoá: nếu lần chạy trước đã os.chdir(REPO) thì cwd đang
# nằm trong thư mục sắp bị xoá, và mọi thao tác mở file sau đó sẽ ném
# FileNotFoundError khó hiểu.
os.chdir(BASE)

# Xoá sạch: extractall KHÔNG tự xoá file cũ, nên một gói cũ có thể "sống sót"
# qua nhiều lần chạy và che mất bản mới.
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]          # bỏ module đã nạp trong bộ nhớ
os.makedirs(REPO, exist_ok=True)

with urllib.request.urlopen(SRC_URL, timeout=60) as r:
    blob = base64.b64decode(r.read())
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    z.extractall(REPO)
print(f'✓ tải mã nguồn từ gist ({len(blob)/1024:.0f} KB)')

miss = missing_in(REPO)
if miss:
    raise SystemExit(f"Gói mã nguồn thiếu module: {', '.join(miss)}")

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

from itercomp import itercomp, load_dataset, normalize_boolean, budget_filter
print('cwd:', os.getcwd(), '| scripts/:', os.path.isdir('scripts'))
print(f'✓ đủ {len(NEED)} module')

In [ ]:
# ══ PREFLIGHT: bản vá cache có chạy trên transformers CỦA KAGGLE không? ══
# SÁU lần vá sai, mỗi lần tốn một phiên GPU, vì bug chỉ lộ ra sau khi đã nạp
# mô hình 7B và chạy 113 giây. Cell này đi ĐÚNG đường code đó trong ~40 giây,
# trước khi tiêu bất cứ thứ gì đắt tiền.
#
# Lần 6 là thất bại IM LẶNG: `_to_legacy` kết thúc bằng `return pkv` khi
# transformers bỏ `to_legacy_cache()`, nên mọi lớp bọc chạy đúng mà chuyển
# tiếp một Cache không đổi.
import torch, transformers

_src = open('scripts/run_eval.py').read()
_ns = {}
exec(_src[_src.index('def _to_legacy'):_src.index('class LLMLinguaMethod')], _ns)

# 1) Cache tổng hợp - 1 giây
print('1)', _ns['selftest_cache_patch']())

# 2) Đường code THẬT: chỉ LongLLMLingua đi qua iterative_compress_prompt,
#    và đó đúng là dòng 1659 đã giết phiên trước.
from llmlingua import PromptCompressor
_c = PromptCompressor(model_name='gpt2',
                      device_map='cuda' if torch.cuda.is_available() else 'cpu')
_ns['_force_legacy_cache'](_c)
_out = _c.compress_prompt(
    ['Hà Nội là thủ đô của Việt Nam. ' * 15,
     'Thành phố Hồ Chí Minh ở miền Nam. ' * 15],
    question='Thủ đô của Việt Nam là gì?', rate=0.5,
    rank_method='longllmlingua', condition_compare=True,
    dynamic_context_compression_ratio=0.3, context_budget='+100',
)['compressed_prompt']
print('2) LongLLMLingua qua iterative_compress_prompt OK:', repr(_out[:70]))

del _c
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print(f'\n✓ PREFLIGHT PASS (transformers {transformers.__version__}) - chạy bảng baseline được.')


In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# ══ BẢNG CHÍNH n=200, READER 7B ══
# ~35 phút trên T4. run_eval.py ghi checkpoint sau MỖI câu nên hết giờ vẫn
# chạy tiếp được từ chỗ dở.
import subprocess, sys, os, time

READER = 'Qwen/Qwen2.5-7B-Instruct'
OUT = 'results/vimqa_200_7b.json'
os.makedirs('results', exist_ok=True)

def run_stream(cmd, logfile, env=None):
    """In output NGAY thay vì giữ tới lúc xong.

    7B ở 4-bit chạy hàng chục phút; capture_output nghĩa là ngồi nhìn màn hình
    trống, không biết treo hay đang chạy.
    """
    t0 = time.time()
    with open(logfile, 'w') as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True,
                             bufsize=1, env=env)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

if os.path.exists(OUT):
    print(f'[bỏ qua] đã có {OUT}')
else:
    run_stream([sys.executable, '-u', 'scripts/run_eval.py',
                '--dataset', 'vimqa', '--limit', '200',
                '--reader', 'hf', '--reader-model', READER, '--load-4bit',
                '--itercomp-llm', 'hf', '--scorer', 'dual',
                '--methods', 'llmlingua2,itercomp',
                '--out', OUT],
               'results/log_vimqa_200_7b.txt')

## Chạy ba baseline, n=200

In [ ]:
# ══ BA BASELINE MỚI ══
import os, sys
READER = 'Qwen/Qwen2.5-7B-Instruct'
os.makedirs('results', exist_ok=True)
OUT = 'results/baselines_vimqa_200_7b.json'

if os.path.exists(OUT):
    print(f'[bỏ qua] {OUT} đã có')
else:
    rc = run_stream(
        [sys.executable, '-u', 'scripts/run_eval.py',
         '--dataset', 'vimqa', '--limit', '200',
         '--reader', 'hf', '--reader-model', READER, '--load-4bit',
         '--itercomp-llm', 'hf', '--scorer', 'dual',
         '--methods', 'llmlingua,longllmlingua,selective-context,'
                          'recomp-extractive,recomp-abstractive',
         '--out', OUT],
        'results/log_baselines.txt')
    assert rc == 0, f'thất bại (rc={rc})' 

## So với bảng đã có

Con số IterCOMP và LLMLingua-2 lấy từ `vimqa_200_7b.json` đã chạy trước.
Nếu ba baseline mới đều dưới IterCOMP thì luận điểm của bài mạnh thêm; nếu
LongLLMLingua bám sát, đó là điều **phải báo cáo** vì nó gần IterCOMP nhất
về ý tưởng.

In [ ]:
import json

new = json.load(open('results/baselines_vimqa_200_7b.json'))['summary']
try:
    ref = json.load(open('results/vimqa_200_7b.json'))['summary']
except FileNotFoundError:
    ref = {}

print(f"{'method':20s} {'F1*':>7s} {'tỉ lệ giữ':>10s}")
for m, v in sorted({**ref, **new}.items(), key=lambda kv: -kv[1]['f1_norm']):
    mark = ' ←mới' if m in new else ''
    print(f"{m:20s} {v['f1_norm']:7.2f} {100*v['ratio']:9.1f}%{mark}")

## Tải về

In [ ]:
import shutil, os
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z = shutil.make_archive(os.path.join(BASE, 'results_baselines'), 'zip', 'results')
print(f'✓ {z}  ({os.path.getsize(z)/1e6:.1f} MB)')